# IRIS-X ARC-AGI-2 Trial 001 — offline Kaggle adapter
Run only after a separate explicit owner authorization. Attach one private Dataset containing the unpacked, provenanced model tree and one private trial-input Dataset, select one NVIDIA RTX PRO 6000 96 GB, and disable Internet before starting.

In [ ]:
from pathlib import Path
import hashlib, json, os, stat, subprocess, sys, zipfile

INPUT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
MAX_FILES = 4097
MAX_FILE = 32 * 1024**3
MAX_TOTAL = 100 * 1024**3
MAX_ARCHIVE = 100 * 1024**3

def digest(data): return 'sha256:' + hashlib.sha256(data).hexdigest()
def file_digest(path):
    hasher = hashlib.sha256()
    with path.open('rb') as stream:
        while chunk := stream.read(1024 * 1024): hasher.update(chunk)
    return 'sha256:' + hasher.hexdigest()

def safe_extract(package, target, expected_kind, expected_digest=None):
    if package.stat().st_size > MAX_ARCHIVE: raise RuntimeError('archive byte limit exceeded')
    archive_digest = file_digest(package)
    if expected_digest is not None and archive_digest != expected_digest:
        raise RuntimeError(f'package digest mismatch: {package}')
    with zipfile.ZipFile(package, 'r') as archive:
        infos = archive.infolist()
        if len(infos) > MAX_FILES: raise RuntimeError('archive file-count limit exceeded')
        names, total = set(), 0
        for info in infos:
            name = info.filename
            parts = name.split('/')
            mode = (info.external_attr >> 16) & 0xffff
            if not name or '\\' in name or name.startswith('/') or any(p in ('', '.', '..') for p in parts): raise RuntimeError(f'unsafe path: {name}')
            if name in names: raise RuntimeError(f'duplicate path: {name}')
            if info.is_dir() or stat.S_ISLNK(mode) or (mode and not stat.S_ISREG(mode)): raise RuntimeError(f'unsafe entry: {name}')
            if info.file_size > MAX_FILE: raise RuntimeError(f'entry byte limit exceeded: {name}')
            if info.compress_size and info.file_size > info.compress_size * 100: raise RuntimeError(f'unsafe expansion ratio: {name}')
            names.add(name); total += info.file_size
            if total > MAX_TOTAL: raise RuntimeError('archive expansion limit exceeded')
        manifest_raw = archive.read('manifest.json')
        manifest = json.loads(manifest_raw)
        canonical = (json.dumps(manifest, ensure_ascii=False, allow_nan=False, sort_keys=True, separators=(',', ':')) + '\n').encode('utf-8')
        if manifest_raw != canonical or manifest.get('format') != 'irisx/closed-world-zip' or manifest.get('format_version') != '1.0': raise RuntimeError('non-canonical or unsupported manifest')
        if manifest['package_kind'] != expected_kind: raise RuntimeError('package kind mismatch')
        declared = {item['path']: item for item in manifest['files']}
        if set(declared) != names - {'manifest.json'}: raise RuntimeError('closed-world membership mismatch')
        target.mkdir(parents=True, exist_ok=False)
        for name, entry in declared.items():
            destination = target.joinpath(*name.split('/'))
            destination.parent.mkdir(parents=True, exist_ok=True)
            hasher, observed = hashlib.sha256(), 0
            with archive.open(name, 'r') as source, destination.open('wb') as sink:
                while chunk := source.read(1024 * 1024):
                    observed += len(chunk); hasher.update(chunk); sink.write(chunk)
            if observed != entry['byte_size'] or 'sha256:' + hasher.hexdigest() != entry['content_digest']: raise RuntimeError(f'file binding mismatch: {name}')
        (target / 'manifest.json').write_bytes(manifest_raw)
    return manifest, archive_digest

trial_packages = list(INPUT.glob('**/trial-input.zip'))
if len(trial_packages) != 1: raise RuntimeError('attach exactly one trial-input.zip')
trial_manifest, trial_digest = safe_extract(trial_packages[0], WORKING / 'irisx-trial', 'irisx/arc-agi-2-trial-input')
expected_trial_digest = input('Enter the preflight-bound trial-input.zip sha256 digest: ').strip()
if trial_digest != expected_trial_digest: raise RuntimeError('trial package does not match preflight')
if file_digest(WORKING / 'irisx-trial/runtime/runner.py') != trial_manifest['metadata']['runner_digest']: raise RuntimeError('runner digest mismatch')
if file_digest(WORKING / 'irisx-trial/runtime/notebook.ipynb') != trial_manifest['metadata']['notebook_digest']: raise RuntimeError('notebook digest mismatch')
provenance_path = WORKING / 'irisx-trial/bindings/model-source-provenance.json'
provenance = json.loads(provenance_path.read_text(encoding='utf-8'))
unsigned = {key: provenance[key] for key in ('schema', 'schema_version', 'artifact_id', 'version', 'payload')}
if provenance.get('schema') != 'irisx/model-source-provenance' or provenance.get('content_digest') != digest(json.dumps(unsigned, ensure_ascii=False, allow_nan=False, sort_keys=True, separators=(',', ':')).encode('utf-8')): raise RuntimeError('model provenance artifact is invalid')
if provenance['content_digest'] != trial_manifest['metadata']['model']['source_provenance_digest']: raise RuntimeError('model provenance binding mismatch')
dataset = trial_manifest['metadata']['model_dataset']
model_root = INPUT / 'datasets' / dataset['owner'] / dataset['slug']
if not model_root.is_dir(): raise RuntimeError(f'bound raw model Dataset is missing: {model_root}')
os.environ['IRISX_KAGGLE_INTERNET_DISABLED'] = '1'
runner = WORKING / 'irisx-trial/runtime/runner.py'
subprocess.run([sys.executable, str(runner), '--trial-root', str(WORKING / 'irisx-trial'), '--model-root', str(model_root), '--output', str(WORKING / 'result-package.zip'), '--trial-package-digest', trial_digest], check=True)
print(WORKING / 'result-package.zip')
